# GazeTR: feature extractor evaluation

Run with the **GazeTR** kernel (env in `envs/GazeTR/`).

There is no ground truth yet, so section 5 checks coverage, plausibility, and stability instead of accuracy. The current test set is `data/test_pictures/`: 15 still images (one-frame clips), 10 of them with labelled gaze targets.

## Notes and verdict

**Hardware used for the speed numbers:** RTX 2080 SUPER 8 GB, CUDA 12.6

**Known from setup (before running on test data):**
- Gaze only: no head pose, no AUs.
- Trained on ETH-XGaze-normalized faces; this notebook feeds a plain face crop instead (no normalization), so accuracy may be lower than the paper's, especially with the camera below the face.
- Output order (yaw/pitch) and signs are not documented.
- No face detector in the repo; YuNet (OpenCV) is used for detection and cropping.
- Code license is non-commercial (CC BY-NC-SA 4.0); weights license not stated.

**Strengths:**

**Weaknesses:**

**Verdict:** keep / drop

**Why:**

## Setup

Imports and paths. Run this first.

In [ ]:
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src import clip_io, plots, schema

ROOT = clip_io.REPO_ROOT
MODEL = 'GazeTR'
MODEL_DIR = ROOT / 'models' / MODEL
WEIGHTS = MODEL_DIR / 'weights' / 'GazeTR-H-ETH.pt'
YUNET_PATH = ROOT / 'models' / 'YuNet' / 'face_detection_yunet_2026may.onnx'
DEVICE = 'cuda'  # model.py hard-codes .cuda() in forward, so CPU won't work

# The repo isn't installable; import model.py from the clone
sys.path.insert(0, str(MODEL_DIR))

## 1. Model card

| Field | Value |
|---|---|
| Paper | Cheng, Lu. *Gaze Estimation using Transformer.* ICPR 2022, arXiv:2105.14424 |
| Repo URL | https://github.com/yihuacheng/GazeTR |
| Commit hash | `d7cf716c9b0f410761ace86aa30439eeca5361da` (2024-05-30) |
| Code license | CC BY-NC-SA 4.0 (README; no LICENSE file): non-commercial, share-alike |
| Weights | `models/GazeTR/weights/GazeTR-H-ETH.pt` (45.8 MB, SHA-256 `5d7da0d42c6147ab7b095ad028fed04f91fb9c218fac2f13cc0e0adc11bc16f8`), from the README's Google Drive link. GazeTR-Hybrid trained on ETH-XGaze, 50 epochs |
| Weights license | **Not stated** |
| Face detector | Not in the repo. OpenCV YuNet, `models/YuNet/face_detection_yunet_2026may.onnx` (229,738 bytes, SHA-256 `ebafce4e3c118d6554634be5c27ab333b4c047a9a8c3faf1d7cf93101c22f0f0`), from https://github.com/opencv/opencv_zoo (`models/face_detection_yunet`), MIT license. Detects faces of about 10–300 px |
| Pipeline | YuNet face box → square crop around the box, resized to 224×224 → ResNet-18 (32 channels, 7×7 map) → 6-layer transformer encoder with a cls token → linear layer → 2 gaze angles |
| Inputs required | 224×224 face, **BGR** (the repo's reader uses `cv2.imread` with no RGB conversion), scaled to [0, 1], no mean/std. Trained on ETH-XGaze-normalized faces; normalization isn't in the repo. **Here: plain face crop, no normalization** |
| Outputs | Gaze, 2 values (gaze2d). **No head pose** (in the dataset labels, but not predicted), **no AUs** |
| Native units / conventions | Radians (ETH-XGaze labels). Order probably [yaw, pitch] (`trainYP.label`; `gtools.gazeto3d` uses output 0 as the horizontal angle), not confirmed; signs undocumented (issues #5, #19) |
| Conversion to schema | **Provisional:** `gaze_yaw_deg` = degrees(output 0), `gaze_pitch_deg` = degrees(output 1), no sign flip. To verify with the gaze pictures. Raw outputs kept as `extra_gaze_raw_0/1`. YuNet score → `face_conf` |
| Env | `envs/GazeTR/` (pyproject.toml + uv.lock), kernel `GazeTR`: Python 3.11.17, torch 2.14.1+cu126 |

In [ ]:
!git -C {MODEL_DIR} remote get-url origin
!git -C {MODEL_DIR} rev-parse HEAD

## 2. Setup log

Ubuntu, uv 0.13.0, Python 3.11.17, torch 2.14.1+cu126 (CUDA runtime 12.6), driver 580.178.04, RTX 2080 SUPER (sm_75).

| # | Step / command | Result | Error | Fix |
|---|---|---|---|---|
| 1 | Inspect repo | No requirements file, not installable, no demo or sample inputs; README "built with pytorch1.7.0" (not a hard pin) | | Use torch 2.x |
| 2 | `uv init --bare --no-workspace --python 3.11`; `uv python pin 3.11` | `envs/GazeTR/` created | | |
| 3 | cu126 index (`explicit = true`) + `uv add torch torchvision` | torch 2.14.1+cu126, torchvision 0.29.1+cu126; CUDA available, sm_75 supported | uv warning: `VIRTUAL_ENV` (OpenFace venv active in shell) ignored | Harmless; `deactivate` |
| 4 | `uv add numpy opencv-python`; `uv add --editable ../..` | numpy 2.4.6, opencv-python 5.0.0.93, pandas 3.0.6, pyarrow 26.0.0, matplotlib 3.11.2 | | Only inference deps; training-only easydict, pyyaml, warmup_scheduler skipped. opencv 5 / pandas 3 differ from OpenFace env (4.11 / 2.2.3) |
| 5 | `uv add --dev ipykernel`; `ipykernel install --user --name GazeTR` | Kernel `GazeTR` | | |
| 6 | Weights downloaded manually to `models/GazeTR/weights/`; `torch.load(..., weights_only=True)` → `Model().load_state_dict` | 205 tensors, no missing or unexpected keys | | |
| 7 | README example: `Model()({'face': torch.ones(10, 3, 224, 224).cuda()})` | Output shape (10, 2), deterministic in eval | | |
| 8 | YuNet model downloaded manually to `models/YuNet/` (2026may version: dynamic input size, needed by OpenCV 5's ONNX engine) | | | |

## 3. Run on test clips

- **Detection:** YuNet on the full frame; only the top-scoring face is used, and only if its score is ≥ 0.5.
- **Crop:** a square of side `CROP_SCALE` × the longer box side, centred on the box, padded with black where it leaves the image, resized to 224×224. No rotation or normalization (see the model card).
- **Model input:** the crop stays BGR and is scaled to [0, 1], as in the repo's reader.

In [ ]:
from model import Model

gazetr = Model().to(DEVICE)
gazetr.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE, weights_only=True))
gazetr.eval()

DET_THRESHOLD = 0.5
# Input size is a placeholder; predict() sets it to each frame's size
detector = cv2.FaceDetectorYN.create(str(YUNET_PATH), '', (320, 320), DET_THRESHOLD)

In [ ]:
CROP_SIZE = 224
CROP_SCALE = 1.0  # crop side relative to the longer side of the detected box

OUTPUT_COLUMNS = (
    ['gaze_yaw_deg', 'gaze_pitch_deg', 'extra_gaze_raw_0', 'extra_gaze_raw_1']
    + ['extra_bbox_x1', 'extra_bbox_y1', 'extra_bbox_x2', 'extra_bbox_y2']
)


def detect(image):
    """Top YuNet face as (score, (x, y, w, h)), or (0.0, None) if there is none."""
    h, w = image.shape[:2]
    detector.setInputSize((w, h))
    _, faces = detector.detect(image)
    if faces is None or len(faces) == 0:
        return 0.0, None
    top = faces[np.argmax(faces[:, -1])]
    return float(top[-1]), top[:4]


def crop_face(image, box):
    """Square crop around the box, padded with black outside the image, resized to CROP_SIZE."""
    x, y, bw, bh = box
    side = CROP_SCALE * max(bw, bh)
    x1, y1 = int(round(x + bw / 2 - side / 2)), int(round(y + bh / 2 - side / 2))
    x2, y2 = x1 + int(round(side)), y1 + int(round(side))
    h, w = image.shape[:2]
    pad = max(0, -x1, -y1, x2 - w, y2 - h)
    if pad:
        image = cv2.copyMakeBorder(image, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)
    crop = image[y1 + pad:y2 + pad, x1 + pad:x2 + pad]
    return cv2.resize(crop, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_AREA)


def predict(image):
    """Run YuNet + GazeTR on one BGR frame. Returns (face_conf, outputs or None)."""
    face_conf, box = detect(image)
    if box is None or face_conf < DET_THRESHOLD:
        return face_conf, None

    crop = crop_face(image, box)
    face = torch.from_numpy(crop).permute(2, 0, 1).float().div(255).unsqueeze(0).to(DEVICE)  # BGR, [0, 1]
    with torch.no_grad():
        gaze = gazetr({'face': face})[0].cpu().numpy()

    x, y, bw, bh = box
    return face_conf, {
        # Provisional: output 0 as yaw, output 1 as pitch, no sign flip; see model card
        'gaze_yaw_deg': float(np.degrees(gaze[0])),
        'gaze_pitch_deg': float(np.degrees(gaze[1])),
        'extra_gaze_raw_0': float(gaze[0]),
        'extra_gaze_raw_1': float(gaze[1]),
        'extra_bbox_x1': float(x), 'extra_bbox_y1': float(y),
        'extra_bbox_x2': float(x + bw), 'extra_bbox_y2': float(y + bh),
    }

In [ ]:
TEST_SET = 'test_pictures'  # folder under data/ with manifest.csv
manifest = clip_io.load_manifest(TEST_SET)
manifest

In [ ]:
results, timings = {}, {}
for row in manifest.itertuples():
    records = []
    t0 = time.perf_counter()
    for frame, timestamp_ms, image in clip_io.iter_frames(clip_io.clip_path(row)):
        face_conf, outputs = predict(image)
        record = {'clip_id': row.clip_id, 'frame': frame, 'timestamp_ms': timestamp_ms,
                  'face_conf': float(face_conf) if outputs is not None else 0.0}
        record.update(outputs or {})
        records.append(record)
    timings[row.clip_id] = time.perf_counter() - t0
    results[row.clip_id] = pd.DataFrame.from_records(records, columns=list(schema.REQUIRED_COLUMNS) + OUTPUT_COLUMNS)
    print(f'{row.clip_id}: {len(records)} frames in {timings[row.clip_id]:.1f} s')

## 4. Save output in the common schema

`save_output` runs `schema.validate` and refuses to write a file that fails it. Fix the conversion in `predict`, not the check.

In [ ]:
for clip_id, df in results.items():
    df = df.astype({'frame': 'int64', 'timestamp_ms': 'float64', 'face_conf': 'float64'})
    out_cols = [c for c in df.columns if c not in schema.REQUIRED_COLUMNS and not c.startswith(schema.EXTRA_PREFIX)]
    df[out_cols] = df[out_cols].astype('float64')
    df = schema.add_missing_au_pairs(df)
    print(clip_io.save_output(df, MODEL, clip_id))

## 5. Quick checks

These read back the saved files, so they test exactly what was written.

In [ ]:
out = clip_io.load_outputs(MODEL)
print('output columns:', schema.output_columns(out))

### Face-detection rate

In [ ]:
display(plots.detection_rate(out).to_frame())
plots.plot_detection_rate(out);

### Speed (fps on this machine)

Includes video decoding and face detection. Note the hardware (CPU/GPU) in the verdict.

In [ ]:
plots.fps_table(timings, out)

### Per-clip summary

One row per clip with mean outputs over detected frames, next to the manifest's gaze target. Raw outputs are shown too, so the yaw/pitch order and signs can be checked against `gaze_target`.

In [ ]:
detected = out[out['face_conf'] >= DET_THRESHOLD]
summary = detected.groupby('clip_id').agg(
    n_frames=('frame', 'size'), face_conf=('face_conf', 'mean'),
    gaze_yaw_deg=('gaze_yaw_deg', 'mean'), gaze_pitch_deg=('gaze_pitch_deg', 'mean'),
    raw_0_deg=('extra_gaze_raw_0', lambda s: np.degrees(s).mean()),
    raw_1_deg=('extra_gaze_raw_1', lambda s: np.degrees(s).mean()),
)
summary = manifest.set_index('clip_id').drop(columns=['path', 'test_set']).join(summary, how='left')
summary.round(2)

### Detections on single-image clips

Top face box, confidence, and gaze arrow drawn on each image, and the 224×224 crop the model saw.

In [ ]:
def draw_gaze(ax, r, color='red'):
    """Gaze arrow from the face box, using the schema columns (so it also checks the sign conversion).
    Assumes a true-orientation image: the subject's left is the image's right."""
    if pd.isna(r['gaze_yaw_deg']):
        return
    x1, y1, x2, y2 = r[['extra_bbox_x1', 'extra_bbox_y1', 'extra_bbox_x2', 'extra_bbox_y2']]
    cx, cy = (x1 + x2) / 2, y1 + 0.4 * (y2 - y1)  # roughly eye level
    length = 1.5 * (x2 - x1)
    yaw, pitch = np.radians(r['gaze_yaw_deg']), np.radians(r['gaze_pitch_deg'])
    dx = length * np.sin(yaw) * np.cos(pitch)  # positive yaw (subject's left) -> image right
    dy = -length * np.sin(pitch)               # positive pitch (up) -> image up
    ax.annotate('', xy=(cx + dx, cy + dy), xytext=(cx, cy),
                arrowprops=dict(arrowstyle='-|>', color=color, lw=2, mutation_scale=15))
    ax.plot(cx, cy, 'o', color=color, ms=3)


still = [r for r in manifest.itertuples() if clip_io.is_image(clip_io.clip_path(r))]
if still:
    ncols = 4
    fig, axes = plt.subplots(-(-len(still) // ncols), ncols, figsize=(3.5 * ncols, 3.5 * -(-len(still) // ncols)), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, row in zip(axes.flat, still):
        _, _, img = next(clip_io.iter_frames(clip_io.clip_path(row), rgb=True))
        r = out[out['clip_id'] == row.clip_id].iloc[0]
        ax.imshow(img)
        title = f"{row.clip_id}  conf={r['face_conf']:.2f}"
        if r['face_conf'] >= DET_THRESHOLD:
            x1, y1, x2, y2 = r[['extra_bbox_x1', 'extra_bbox_y1', 'extra_bbox_x2', 'extra_bbox_y2']]
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime', lw=1.5))
            draw_gaze(ax, r)
            title += f"\nyaw={r['gaze_yaw_deg']:.1f}  pitch={r['gaze_pitch_deg']:.1f}"
        ax.set_title(title, fontsize='small')
    plt.tight_layout()
    plt.show()

In [ ]:
# The crops the model actually saw (shown in RGB; the model gets BGR)
if still:
    ncols = 8
    fig, axes = plt.subplots(-(-len(still) // ncols), ncols, figsize=(1.8 * ncols, 2 * -(-len(still) // ncols)), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, row in zip(axes.flat, still):
        _, _, img = next(clip_io.iter_frames(clip_io.clip_path(row)))
        face_conf, box = detect(img)
        if box is not None and face_conf >= DET_THRESHOLD:
            ax.imshow(cv2.cvtColor(crop_face(img, box), cv2.COLOR_BGR2RGB))
        ax.set_title(row.clip_id, fontsize='x-small')
    plt.tight_layout()
    plt.show()

### Standard plots per clip

Video clips only (time series need more than one frame).

In [ ]:
for clip_id, df in out.groupby('clip_id'):
    if len(df) < 2:
        continue
    plots.plot_face_conf(df)
    plots.plot_gaze(df)
    plt.show()

### Plausibility and stability (no ground truth)

- Value ranges: do the angles look reasonable for what happens in each clip?
- Jitter: median absolute frame-to-frame change on detected frames. A high value on a still face means a noisy model.

In [ ]:
angle_cols = [c for c in schema.GAZE_COLUMNS if c in out]
display(detected[angle_cols].describe().T.round(3))

# Jitter needs consecutive frames, so only video clips count
multi = detected.groupby('clip_id').filter(lambda d: len(d) > 1)
if len(multi):
    jitter = multi.groupby('clip_id')[angle_cols].apply(lambda d: d.diff().abs().median())
    display(jitter.rename(columns=lambda c: c.replace('_deg', '_jitter_deg')))
else:
    print('no multi-frame clips yet: jitter skipped')

### Failure cases

Frames where the face was missed or detected with low confidence. Look at a few to see why (pose, occlusion, lighting, blur).

In [ ]:
failures = plots.failure_frames(out, threshold=DET_THRESHOLD)
display(failures.groupby('clip_id').size().rename('n_failures').to_frame())
failures.head(20)

In [ ]:
def show_frames(clip_id, frames, ncols=4):
    row = manifest.set_index('clip_id').loc[clip_id]
    wanted = set(frames)
    images = [(f, img) for f, _, img in clip_io.iter_frames(clip_io.clip_path(row), rgb=True) if f in wanted]
    if not images:
        return
    nrows = -(-len(images) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 2.5 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (f, img) in zip(axes.flat, images):
        ax.imshow(img)
        ax.set_title(f'{clip_id} frame {f}', fontsize='small')
    plt.show()

# Up to 8 evenly spaced failure frames per clip
for clip_id, f in failures.groupby('clip_id'):
    show_frames(clip_id, f['frame'].iloc[np.linspace(0, len(f) - 1, min(8, len(f))).astype(int)])